In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize

import mrfitty
from mrfitty.base import (
    AdaptiveEnergyRangeBuilder,
    InterpolatedReferenceSpectraSet,
    ReferenceSpectrum,
)

%matplotlib inline

In [ ]:
import fnmatch


def filter_spectra_by_name(spectra_list, *patterns):
    """Return spectra whose file_name matches any of the glob-style patterns."""
    matches = [s for s in spectra_list if any(fnmatch.fnmatch(s.file_name, p) for p in patterns)]
    if not matches:
        raise ValueError(
            f"No spectra matched the given pattern(s): {patterns!r}. "
            f"Available file names: {[s.file_name for s in spectra_list]}"
        )
    return matches


In [ ]:
src_path, _ = os.path.split(mrfitty.__path__[0])
sample_data_dir_path = os.path.join(src_path, 'example', 'arsenic')
print('sample data is installed at "{}"'.format(sample_data_dir_path))
os.path.exists(sample_data_dir_path)

In [ ]:
sample_data_reference_glob = os.path.join(sample_data_dir_path, 'reference/*.e')
print('sample data reference glob: {}'.format(sample_data_reference_glob))
sample_data_unknown_glob = os.path.join(sample_data_dir_path, 'unknown/*.e')
print('sample data unknown glob: {}'.format(sample_data_unknown_glob))

In [ ]:
sample_data_reference_set, _ = list(ReferenceSpectrum.read_all([sample_data_reference_glob]))
sample_data_reference_list = sorted(list(sample_data_reference_set), key=lambda s: s.file_name)
print('sample data reference file count: {}'.format(len(sample_data_reference_list)))
sample_data_unknown_set, _ = list(ReferenceSpectrum.read_all([sample_data_unknown_glob]))
sample_data_unknown_list = sorted(list(sample_data_unknown_set), key=lambda s: s.file_name)
print('sample data unknown file count: {}'.format(len(sample_data_unknown_list)))

In [ ]:
unknown_spectrum = filter_spectra_by_name(sample_data_unknown_list, "OTT3_55*")[0]
print(f'unknown spectrum: {unknown_spectrum.file_name}')
reference_spectra = filter_spectra_by_name(
    sample_data_reference_list,
    "Arsenopyrite_Jul*", "orpiment_all*", "arsenate*_diop*") 

interp_ref_set = InterpolatedReferenceSpectraSet(
    unknown_spectrum=unknown_spectrum,
    reference_set=reference_spectra,
)

subset = interp_ref_set.get_reference_subset_and_unknown_df(
    reference_list=reference_spectra,
    energy_range_builder=AdaptiveEnergyRangeBuilder(),
)

A = subset['reference_subset_df'].values
b = subset['unknown_subset_df']['norm'].values
energies = subset['reference_subset_df'].index.values
ref_names = list(subset['reference_subset_df'].columns)

coef, _ = scipy.optimize.nnls(A, b)
fitted = A @ coef

print('Coefficients:')
for ref_coef, ref_name in sorted(zip(coef, ref_names), reverse=True):
    print(f"{ref_coef:0.6f}: {ref_name}")

In [ ]:
residuals = fitted - b
rmse = np.sqrt(np.mean(np.square(residuals)))
print(f'Residuals: n={len(residuals)}, mean={residuals.mean():.6f}, std={residuals.std():.6f}')
print(f'RMSE: {rmse:.6f}')

In [ ]:
def plot_spectrum_fit(energies, b, fitted, residuals, ax):
    ax.plot(energies, b, label='unknown', color='black')
    ax.plot(energies, fitted, label='fit', color='red', linestyle='--')
    ax.scatter(energies, residuals, color='orange', label='residuals', marker='o', s=10.0)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Normalized Fluorescence')
    ax.legend()


fig, ax_fit = plt.subplots(figsize=(10, 4))
plot_spectrum_fit(energies, b, fitted, residuals, ax=ax_fit)
ax_fit.set_title(f'NNLS Fit: {unknown_spectrum.file_name}')
plt.tight_layout()
plt.show()

In [ ]:
n = len(residuals)
n_lags = min(40, n // 2)
x = residuals - residuals.mean()
var = np.dot(x, x) / n
acf_values = np.array(
    [1.0] + [np.dot(x[:-lag], x[lag:]) / (n * var) for lag in range(1, n_lags + 1)]
)
lags = np.arange(n_lags + 1)

print(f'ACF at lag 0: {acf_values[0]:.4f}')
print(f'ACF at lag 1: {acf_values[1]:.4f}')
print(f'ACF at lag 2: {acf_values[2]:.4f}')
print(f'ACF at lag 5: {acf_values[5]:.4f}')

In [ ]:
def plot_acf(lags, acf_values, n, ax):
    ci_95 = 1.96 / np.sqrt(n)
    ax.bar(lags, acf_values, color='steelblue', alpha=0.7)
    ax.axhline(ci_95, color='red', linestyle='--', label='95% CI (white noise)')
    ax.axhline(-ci_95, color='red', linestyle='--')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.legend()


fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(lags, acf_values, n, ax=ax)
ax.set_title('Autocorrelation Function of Residuals')
plt.tight_layout()
plt.show()

In [ ]:
def moving_block_holdout_bootstrap(A, b, fitted, residuals, rng, n_bootstrap=1000):
    n = len(residuals)
    block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    # assume n=198 and block_length=6
    # then block_starts looks like
    #  array([  0,   1,   2, ..., 192 ])
    block_starts = np.arange(n - block_length + 1)

    bootstrap_coefs = np.zeros((n_bootstrap, A.shape[1]))
    bootstrap_pes = np.zeros(n_bootstrap)
    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)

    for i in range(n_bootstrap):
        # Randomly select non-contiguous holdout blocks totaling ~1/3 of the data
        # holdout_block_indices look like
        #  array([ 6, 20,  8,  3, 16, 28, 24,  2, 32, 22, 18])
        holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            holdout_mask[idx * block_length:(idx + 1) * block_length] = True
        holdout_masks[i] = holdout_mask
        train_mask = ~holdout_mask

        # Restrict bootstrap block starts to positions that don't overlap any holdout block
        valid_block_starts = block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
        ]

        # Build bootstrap sample from moving blocks of residuals (holdout positions excluded)
        sampled_starts = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)
        bootstrap_residuals = np.concatenate(
            [residuals[s:s + block_length] for s in sampled_starts]
        )[:n]
        bootstrap_b = fitted + bootstrap_residuals

        # Fit on bootstrap data with holdout blocks removed
        bootstrap_coef, _ = scipy.optimize.nnls(A[train_mask], bootstrap_b[train_mask])
        bootstrap_coefs[i] = bootstrap_coef

        # Prediction error on real data at the held-out positions
        holdout_residuals = A[holdout_mask] @ bootstrap_coef - b[holdout_mask]
        bootstrap_pes[i] = np.sqrt(np.mean(np.square(holdout_residuals)))

    return bootstrap_coefs, bootstrap_pes, block_length, n_holdout_blocks, holdout_masks


rng = np.random.default_rng(seed=42)
n_bootstrap = 1000

bootstrap_coefs, bootstrap_pes, block_length, n_holdout_blocks, holdout_masks = \
    moving_block_holdout_bootstrap(A, b, fitted, residuals, rng, n_bootstrap=n_bootstrap)

n = len(residuals)
print(f'n={n}, block_length={block_length}, n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data)')
print(f'Coefficient means: {bootstrap_coefs.mean(axis=0)}')
print(f'Coefficient stds:  {bootstrap_coefs.std(axis=0)}')
print(f'Prediction error mean={bootstrap_pes.mean():.6f}, std={bootstrap_pes.std():.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(energies, b, label='unknown spectrum', color='black', linewidth=1.0)
ax.scatter(
    energies[~holdout_masks[-1]], b[~holdout_masks[-1]],
    color='steelblue', s=10, zorder=4, alpha=0.6,
    label=f'training ({(~holdout_masks[-1]).sum()} points)',
)
ax.scatter(
    energies[holdout_masks[-1]], b[holdout_masks[-1]],
    color='orange', s=20, zorder=5,
    label=f'holdout ({holdout_masks[-1].sum()} points)',
)

ax.set_xlabel('Energy (eV)')
ax.set_ylabel('Normalized Fluorescence')
ax.set_title(
    f'Last Holdout Mask — {unknown_spectrum.file_name}\n'
    f'block_length={block_length}, n_holdout_blocks={n_holdout_blocks}'
)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_bootstrap_summary(
    energies, b, fitted, residuals,
    lags, acf_values,
    bootstrap_coefs, bootstrap_pes,
    coef, ref_names,
    spectrum_name, n_bootstrap,
):
    import matplotlib.gridspec as gridspec

    n = len(residuals)
    n_refs = len(ref_names)
    n_cols = n_refs + 1
    rmse = np.sqrt(np.mean(residuals ** 2))

    fig = plt.figure(figsize=(4 * n_cols, 16))
    gs = gridspec.GridSpec(4, n_cols, figure=fig)

    ax_fit = fig.add_subplot(gs[0, :])
    ax_acf = fig.add_subplot(gs[1, :])
    axs = np.array([[fig.add_subplot(gs[row, col]) for col in range(n_cols)] for row in range(2, 4)])

    plot_spectrum_fit(energies, b, fitted, residuals, ax=ax_fit)
    ax_fit.set_title(spectrum_name)

    plot_acf(lags, acf_values, n, ax=ax_acf)
    ax_acf.set_title('Residual Autocorrelation Function')
    ax_acf.legend(fontsize=12)

    for j, (coef_mean, coef_i, name) \
            in enumerate(sorted(zip(bootstrap_coefs.mean(axis=0), range(n_refs), ref_names), reverse=True)):

        p2_5 = np.percentile(bootstrap_coefs[:, coef_i], 2.5)
        p97_5 = np.percentile(bootstrap_coefs[:, coef_i], 97.5)

        axs[0, j].hist(bootstrap_coefs[:, coef_i], bins=40, color='steelblue', alpha=0.7, edgecolor='white')
        axs[0, j].axvline(coef[coef_i], color='red', linestyle='--', label=f'observed={coef[coef_i]:.3f}')
        axs[0, j].axvline(coef_mean, color='blue', linestyle='--', label=f'mean={coef_mean:.3f}')
        axs[0, j].axvline(p2_5, color='green', linestyle=':', label=f'2.5%={p2_5:.3f}')
        axs[0, j].axvline(p97_5, color='green', linestyle=':', label=f'97.5%={p97_5:.3f}')
        axs[0, j].set_title(name, fontsize=9)
        axs[0, j].set_xlabel('Coefficient')
        axs[0, j].legend(fontsize=8)

        axs[1, j].violinplot(bootstrap_coefs[:, coef_i])
        axs[1, j].scatter(
            [0.95], [coef[coef_i]], color='red', zorder=5, marker='o', s=60,
            edgecolors='black', linewidths=0.8, label=f'observed={coef[coef_i]:.3f}',
        )
        axs[1, j].scatter(
            [1.05], [coef_mean], color='blue', zorder=5, marker='D', s=60,
            edgecolors='black', linewidths=0.8, label=f'mean={coef_mean:.3f}',
        )
        axs[1, j].set_title(name, fontsize=9)
        axs[1, j].legend(fontsize=8)

    pes_mean = bootstrap_pes.mean()
    pes_p2_5 = np.percentile(bootstrap_pes, 2.5)
    pes_p97_5 = np.percentile(bootstrap_pes, 97.5)

    axs[0, n_refs].hist(bootstrap_pes, bins=40, color='darkorange', alpha=0.7, edgecolor='white')
    axs[0, n_refs].axvline(rmse, color='red', linestyle='--', label=f'RMSE={rmse:.4f}')
    axs[0, n_refs].axvline(pes_mean, color='blue', linestyle='--', label=f'mean={pes_mean:.4f}')
    axs[0, n_refs].axvline(pes_p2_5, color='green', linestyle=':', label=f'2.5%={pes_p2_5:.4f}')
    axs[0, n_refs].axvline(pes_p97_5, color='green', linestyle=':', label=f'97.5%={pes_p97_5:.4f}')
    axs[0, n_refs].set_title('Holdout Prediction Error')
    axs[0, n_refs].set_xlabel('Prediction Error (RMSE)')
    axs[0, n_refs].legend(fontsize=8)

    parts = axs[1, n_refs].violinplot(bootstrap_pes)
    for pc in parts['bodies']:
        pc.set_facecolor('darkorange')
        pc.set_alpha(0.7)
    axs[1, n_refs].scatter(
        [0.95], [rmse], color='red', zorder=5, marker='o', s=60,
        edgecolors='black', linewidths=0.8, label=f'RMSE={rmse:.4f}',
    )
    axs[1, n_refs].scatter(
        [1.05], [pes_mean], color='blue', zorder=5, marker='D', s=60,
        edgecolors='black', linewidths=0.8, label=f'mean={pes_mean:.4f}',
    )
    axs[1, n_refs].set_title('Holdout Prediction Error')
    axs[1, n_refs].legend(fontsize=8)

    coef_violin_axes = axs[1, :n_refs]
    all_ylims = [ax.get_ylim() for ax in coef_violin_axes]
    global_ymin = min(lo for lo, hi in all_ylims)
    global_ymax = max(hi for lo, hi in all_ylims)
    for ax in coef_violin_axes:
        ax.set_ylim(global_ymin, global_ymax)

    plt.suptitle(f'Moving Block Holdout Bootstrap Distributions ({n_bootstrap} iterations)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_bootstrap_summary(
    energies, b, fitted, residuals,
    lags, acf_values,
    bootstrap_coefs, bootstrap_pes,
    coef, ref_names,
    spectrum_name=unknown_spectrum.file_name,
    n_bootstrap=n_bootstrap,
)

In [ ]:
def interpolate_references_at_sample_energies(reference_spectra, spectrum):
    energies = spectrum.data_df.index.values
    print(f'spectrum: {spectrum.file_name}')
    print(f'energy range: {energies[0]:.2f}–{energies[-1]:.2f} eV ({len(energies)} points)')
    print(f'references: {len(reference_spectra)}')

    # restrict to energies covered by the spectrum AND every reference to avoid extrapolation
    energy_min = max(
        energies[0],
        max(ref.data_df.index.values[0] for ref in reference_spectra),
    )
    energy_max = min(
        energies[-1],
        min(ref.data_df.index.values[-1] for ref in reference_spectra),
    )
    valid_mask = (energies >= energy_min) & (energies <= energy_max)
    valid_energies = energies[valid_mask]
    n_excluded = (~valid_mask).sum()
    if n_excluded:
        print(f'excluded {n_excluded} energies outside common range '
              f'({energy_min:.2f}–{energy_max:.2f} eV)')
    print(f'interpolating at {len(valid_energies)} energies '
          f'({valid_energies[0]:.2f}–{valid_energies[-1]:.2f} eV)')

    result = np.zeros((len(valid_energies), len(reference_spectra)))
    for i, ref in enumerate(reference_spectra):
        result[:, i] = ref.interpolant(valid_energies)
        print(f'  {ref.file_name}: norm [{result[:, i].min():.4f}, {result[:, i].max():.4f}]')

    return valid_energies, result

In [ ]:
energies, A = interpolate_references_at_sample_energies(reference_spectra, unknown_spectrum)

In [ ]:
energies.shape

In [ ]:
A.shape

In [ ]:
def select_holdout_blocks(n, rng, n_bootstrap=1000):
    """Pre-compute holdout masks and bootstrap block samples for n_bootstrap iterations.

    Captures all randomness so the same draws can be reused across multiple
    reference subsets via do_moving_block_holdout_bootstrap.

    Returns
    -------
    holdout_masks   : (n_bootstrap, n) bool array
    sampled_starts  : (n_bootstrap, n_blocks_needed) int array — block start indices for resampling
    block_length    : int
    n_holdout_blocks : int
    """
    block_length = max(1, int(np.round(n ** (1 / 3))))
    n_blocks_needed = int(np.ceil(n / block_length))
    n_full_blocks = n // block_length
    n_holdout_blocks = round(n_full_blocks / 3)
    block_starts = np.arange(n - block_length + 1)

    print(f'n={n}, block_length={block_length}, n_full_blocks={n_full_blocks}')
    print(f'n_holdout_blocks={n_holdout_blocks} (~{n_holdout_blocks * block_length / n:.0%} of data), '
          f'n_blocks_needed={n_blocks_needed}')

    holdout_masks = np.zeros((n_bootstrap, n), dtype=bool)
    sampled_starts = np.zeros((n_bootstrap, n_blocks_needed), dtype=int)

    for i in range(n_bootstrap):
        holdout_block_indices = rng.choice(n_full_blocks, size=n_holdout_blocks, replace=False)
        holdout_mask = np.zeros(n, dtype=bool)
        for idx in holdout_block_indices:
            holdout_mask[idx * block_length:(idx + 1) * block_length] = True
        holdout_masks[i] = holdout_mask

        valid_block_starts = block_starts[
            np.array([not holdout_mask[s:s + block_length].any() for s in block_starts])
        ]
        sampled_starts[i] = rng.choice(valid_block_starts, size=n_blocks_needed, replace=True)

    return holdout_masks, sampled_starts, block_length, n_holdout_blocks


def do_moving_block_holdout_bootstrap(A, b, holdout_masks, sampled_starts, block_length):
    """Fit NNLS and compute holdout prediction error for each pre-drawn bootstrap iteration.

    Parameters
    ----------
    A               : (n, n_refs) design matrix for one reference subset
    b               : (n,) observed spectrum values
    holdout_masks   : (n_bootstrap, n) bool array from select_holdout_blocks
    sampled_starts  : (n_bootstrap, n_blocks_needed) int array from select_holdout_blocks
    block_length    : int from select_holdout_blocks

    Returns
    -------
    bootstrap_coefs : (n_bootstrap, n_refs) float array
    bootstrap_pes   : (n_bootstrap,) float array — per-iteration holdout RMSE
    coef            : (n_refs,) float array — full-data NNLS coefficients
    fitted          : (n,) float array — full-data fitted values
    residuals       : (n,) float array — full-data residuals
    lags            : (n_lags+1,) int array — ACF lag indices
    acf_values      : (n_lags+1,) float array — ACF of full-data residuals
    """
    n = len(b)
    n_bootstrap = len(holdout_masks)
    bootstrap_coefs = np.zeros((n_bootstrap, A.shape[1]))
    bootstrap_pes = np.zeros(n_bootstrap)

    coef, _ = scipy.optimize.nnls(A, b)
    fitted = A @ coef
    residuals = fitted - b

    n_lags = min(40, n // 2)
    x = residuals - residuals.mean()
    var = np.dot(x, x) / n
    acf_values = np.array(
        [1.0] + [np.dot(x[:-lag], x[lag:]) / (n * var) for lag in range(1, n_lags + 1)]
    )
    lags = np.arange(n_lags + 1)

    for i in range(n_bootstrap):
        holdout_mask = holdout_masks[i]
        train_mask = ~holdout_mask

        bootstrap_residuals = np.concatenate(
            [residuals[s:s + block_length] for s in sampled_starts[i]]
        )[:n]
        bootstrap_b = fitted + bootstrap_residuals

        bootstrap_coef, _ = scipy.optimize.nnls(A[train_mask], bootstrap_b[train_mask])
        bootstrap_coefs[i] = bootstrap_coef

        holdout_residuals = A[holdout_mask] @ bootstrap_coef - b[holdout_mask]
        bootstrap_pes[i] = np.sqrt(np.mean(np.square(holdout_residuals)))

    return bootstrap_coefs, bootstrap_pes, coef, fitted, residuals, lags, acf_values

In [ ]:
from itertools import combinations
from math import comb


def do_ref_subset_moving_block_holdout_bootstrap(b, A, M, rng, n_bootstrap=1000):
    """Bootstrap prediction error for every combination of references at each size in M.

    select_holdout_blocks is called once and its draws are shared across all
    combinations so that per-combination prediction errors are directly comparable.

    Parameters
    ----------
    b           : (n,) observed spectrum normalized fluorescence values
    A           : (n, n_refs) interpolated reference spectrum fluorescence values
    M           : list of combination sizes to evaluate (each between 1 and n_refs)
    rng         : numpy random Generator
    n_bootstrap : number of bootstrap iterations

    Returns
    -------
    results : dict of numpy arrays, one row per combination across all sizes in M:
        'M'               : (n_combinations,) int
        'ref_indices'     : (n_combinations, max_M) float — column indices into A, NaN for unused
        'bootstrap_coefs' : (n_combinations, n_bootstrap, max_M) float — NaN for unused coefficients
        'bootstrap_pes'   : (n_combinations, n_bootstrap) float
        'coef'            : (n_combinations, max_M) float — full-data NNLS coefficients, NaN for unused
        'fitted'          : (n_combinations, n) float
        'residuals'       : (n_combinations, n) float
        'lags'            : (n_lags+1,) int — ACF lag indices (shared across all combinations)
        'acf_values'      : (n_combinations, n_lags+1) float — ACF of full-data residuals
    holdout_masks    : (n_bootstrap, n) bool array
    sampled_starts   : (n_bootstrap, n_blocks_needed) int array
    block_length     : int
    n_holdout_blocks : int
    """
    n, n_refs = A.shape
    for m in M:
        if not (1 <= m <= n_refs):
            raise ValueError(f'M value {m} must be between 1 and n_refs={n_refs}')

    holdout_masks, sampled_starts, block_length, n_holdout_blocks = \
        select_holdout_blocks(n, rng, n_bootstrap=n_bootstrap)

    max_M = max(M)
    n_combinations = sum(comb(n_refs, m) for m in M)
    n_lags = min(40, n // 2)

    all_M = np.zeros(n_combinations, dtype=int)
    all_ref_indices = np.full((n_combinations, max_M), np.nan)
    all_bootstrap_coefs = np.full((n_combinations, n_bootstrap, max_M), np.nan)
    all_bootstrap_pes = np.zeros((n_combinations, n_bootstrap))
    all_coef = np.full((n_combinations, max_M), np.nan)
    all_fitted = np.zeros((n_combinations, n))
    all_residuals = np.zeros((n_combinations, n))
    all_acf_values = np.zeros((n_combinations, n_lags + 1))

    i = 0
    for m in sorted(M):
        for ref_indices in combinations(range(n_refs), m):
            A_sub = A[:, ref_indices]
            bootstrap_coefs, bootstrap_pes, coef, fitted, residuals, lags, acf_values = \
                do_moving_block_holdout_bootstrap(A_sub, b, holdout_masks, sampled_starts, block_length)

            all_M[i] = m
            all_ref_indices[i, :m] = ref_indices
            all_bootstrap_coefs[i, :, :m] = bootstrap_coefs
            all_bootstrap_pes[i] = bootstrap_pes
            all_coef[i, :m] = coef
            all_fitted[i] = fitted
            all_residuals[i] = residuals
            all_acf_values[i] = acf_values
            i += 1

    results = {
        'M': all_M,
        'ref_indices': all_ref_indices,
        'bootstrap_coefs': all_bootstrap_coefs,
        'bootstrap_pes': all_bootstrap_pes,
        'coef': all_coef,
        'fitted': all_fitted,
        'residuals': all_residuals,
        'lags': lags,
        'acf_values': all_acf_values,
    }
    return results, holdout_masks, sampled_starts, block_length, n_holdout_blocks

In [ ]:
def plot_ref_subsets_summary(results, ref_names, spectrum_name):
    """Violin plot of bootstrap prediction errors for every reference subset.

    Subsets are sorted in descending order by mean prediction error.

    Parameters
    ----------
    results     : dict returned by do_ref_subset_moving_block_holdout_bootstrap
    ref_names   : list of reference spectrum names (indexed by results['ref_indices'])
    spectrum_name : str — used in the plot title
    """
    from matplotlib.patches import Patch

    n_combinations = len(results['M'])
    unique_M = sorted(set(results['M']))
    colors = plt.cm.tab10(np.linspace(0, 0.4, len(unique_M)))
    M_to_color = {m: c for m, c in zip(unique_M, colors)}

    order = np.argsort(results['bootstrap_pes'].mean(axis=1))[::-1]

    labels = []
    for i in order:
        m = results['M'][i]
        ref_idx = [int(j) for j in results['ref_indices'][i, :m]]
        short_names = [ref_names[j].rsplit('.', 1)[0] for j in ref_idx]
        labels.append('\n+ '.join(short_names))

    fig, ax = plt.subplots(figsize=(max(8, n_combinations * 2), 5))

    parts = ax.violinplot(
        [results['bootstrap_pes'][i] for i in order],
        positions=range(n_combinations),
        showmedians=True,
    )
    for plot_pos, i in enumerate(order):
        parts['bodies'][plot_pos].set_facecolor(M_to_color[results['M'][i]])
        parts['bodies'][plot_pos].set_alpha(0.7)

    ax.set_xticks(range(n_combinations))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('Holdout Prediction Error (RMSE)')
    ax.set_xlabel('Reference Subset')
    ax.set_title(f'Bootstrap Prediction Error by Reference Subset — {spectrum_name}')

    legend_elements = [
        Patch(facecolor=M_to_color[m], alpha=0.7, label=f'M={m}') for m in unique_M
    ]
    ax.legend(handles=legend_elements)

    plt.tight_layout()
    plt.show()

In [ ]:
rng3 = np.random.default_rng(seed=42)
n_bootstrap = 1000

results, holdout_masks3, sampled_starts3, block_length3, n_holdout_blocks3 = \
    do_ref_subset_moving_block_holdout_bootstrap(b, A, M=[1, 2, 3], rng=rng3, n_bootstrap=n_bootstrap)

plot_ref_subsets_summary(results, ref_names, spectrum_name=unknown_spectrum.file_name)

for i in range(len(results['M'])):
    m = results['M'][i]
    ref_idx = [int(j) for j in results['ref_indices'][i, :m]]
    subset_ref_names = [ref_names[j] for j in ref_idx]

    plot_bootstrap_summary(
        energies, b,
        results['fitted'][i],
        results['residuals'][i],
        results['lags'],
        results['acf_values'][i],
        results['bootstrap_coefs'][i, :, :m],
        results['bootstrap_pes'][i],
        results['coef'][i, :m],
        subset_ref_names,
        spectrum_name=unknown_spectrum.file_name,
        n_bootstrap=n_bootstrap,
    )